# 刻蚀虚拟量测（Virtual Metrology）建模比赛

## 比赛目标

利用传感器数据（OES 光谱、RF 信号、压力、温度、气体流量等），预测 SiO₂ 等离子体刻蚀速率（nm/min）。

**评估指标：**
- R² Score（60%）
- MAE（20%）
- 方法论评审（20%）— 分析质量、改进思路、代码可读性

## 背景

在半导体制造中，每个晶圆的刻蚀速率通常只能通过离线量测（metrology）获得，采样率仅约 10-15%。虚拟量测（VM）通过实时传感器数据预测量测结果，使每一片晶圆都获得"虚拟"的量测值。

**推荐观看：**
- [Applied Materials 刻蚀设备介绍](https://www.youtube.com/watch?v=J-1MqRm7RbI)
- [Virtual Metrology 概念介绍](https://www.youtube.com/watch?v=GqHmROJ8Cz4)

## 数据说明

| 数据集 | 样本数 | 来源 | 说明 |
|--------|--------|------|------|
| `train.csv` | 1500 | Chamber_1, lots 1-60 | 含 etch_rate 标签 |
| `test_features.csv` | 1000 | Chamber_1+2, lots 61-80 | 无标签，用于提交预测 |

**注意：** 测试集包含跨腔体（Chamber_2）和时间段偏移（lots 61-80），分布与训练集有所不同。

## 提交格式

提交 CSV 文件，包含两列：`run_id` 和 `etch_rate`（预测值）。
## 關於

Uah sorry, this starter notebook is written in Chinese, please use proper models to translate to English and other target languages if needed.

## Block 1: 业务背景（面向 ML 工程师）

### 什么是等离子体刻蚀？

等离子体刻蚀是半导体制造中将图案从光刻胶转移到硅片表面的关键工艺。在电容耦合等离子体（CCP）反应腔中：

1. **RF 功率** 电离工艺气体（CF₄/Ar/CHF₃），产生含氟自由基
2. 氟自由基（F*）与 SiO₂ 发生化学反应，生成挥发性 SiF₄
3. **离子轰击** 增强化学反应速率，实现各向异性刻蚀

### 为什么需要虚拟量测？

| | 传统量测 | 虚拟量测 |
|---|---------|---------|
| 覆盖率 | 10-15% 晶圆 | 100% 晶圆 |
| 延迟 | 数小时 | 实时 |
| 成本 | 高（破坏性测试） | 低（利用已有传感器） |

### 本比赛仿真环境的物理模型

仿真基于 CCP 刻蚀反应腔的物理模型，包含：
- **非线性功率-压力耦合**：功率和压力的交互影响氟自由基密度
- **非单调腔体老化**：PM（预防性维护）后的 break-in 阶段，刻蚀速率先升后降
- **传感器噪声**：OES 散粒噪声、RF 测量噪声、MFC 校准漂移
- **未观测混淆因子**：基板变异（2%）和批次偏移（1.5%），造成 R² 上限约 0.96

In [ ]:
# Block 2: 数据加载与概览
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 5)

# 加载数据
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test_features.csv')
feat_desc = pd.read_csv('data/feature_description.csv')

print(f'训练集: {train.shape}')
print(f'测试集: {test.shape}')
print(f'特征数: {len(feat_desc)}')
print()

# 元数据列
meta_cols = ['run_id', 'chamber_id', 'lot_id', 'wafer_id', 'is_metrology', 'after_pm']
feature_cols = [c for c in train.columns if c not in meta_cols + ['etch_rate']]
print(f'特征列数: {len(feature_cols)}')
print(f'元数据列: {meta_cols}')
print()

# 目标变量分布
print(f"刻蚀速率统计:")
print(train['etch_rate'].describe())
print(f"\n量测采样率: {train['is_metrology'].mean():.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train['etch_rate'], bins=50, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Etch Rate (nm/min)')
axes[0].set_ylabel('Count')
axes[0].set_title('训练集刻蚀速率分布')
axes[0].axvline(train['etch_rate'].mean(), color='red', linestyle='--', label=f"Mean={train['etch_rate'].mean():.0f}")
axes[0].legend()

axes[1].bar(['Train', 'Test'], [len(train), len(test)], color=['steelblue', 'coral'])
axes[1].set_ylabel('样本数')
axes[1].set_title('训练集 vs 测试集样本量')
for i, v in enumerate([len(train), len(test)]):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Block 3: 特征探索

# 特征分类统计
cat_counts = feat_desc['category'].value_counts()
print('特征分类:')
for cat, cnt in cat_counts.items():
    print(f'  {cat}: {cnt} 个特征')

# Top-15 特征与 etch_rate 的相关性
correlations = train[feature_cols + ['etch_rate']].corr()['etch_rate'].drop('etch_rate').abs().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 相关性 bar chart
top15 = correlations.head(15)
colors = {'OES': '#e74c3c', 'OES_Ratio': '#e67e22', 'RF': '#2ecc71',
          'Process': '#3498db', 'Cross': '#9b59b6', 'Chamber': '#1abc9c'}
bar_colors = [colors.get(feat_desc[feat_desc['feature']==f]['category'].values[0], '#95a5a6') for f in top15.index]
axes[0].barh(range(15), top15.values[::-1], color=bar_colors[::-1])
axes[0].set_yticks(range(15))
axes[0].set_yticklabels(top15.index[::-1])
axes[0].set_xlabel('|Correlation with etch_rate|')
axes[0].set_title('Top 15 特征与刻蚀速率的相关性')

# 图例
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k) for k, c in colors.items()]
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=9)

# 特征分类 pie chart
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.0f%%',
            colors=['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6', '#1abc9c'])
axes[1].set_title(f'特征类别分布（共 {len(feature_cols)} 个特征）')

plt.tight_layout()
plt.show()

print(f'\n最高相关性特征: {top15.index[0]} (r={top15.values[0]:.3f})')
print(f'最低相关性特征: {correlations.index[-1]} (r={correlations.values[-1]:.3f})')

In [ ]:
# Block 4: Chamber Aging 可视化

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# 训练集 etch rate time-series
ax = axes[0]
ax.plot(range(len(train)), train['etch_rate'].values, alpha=0.4, linewidth=0.8, color='steelblue')
# 标注 PM 事件
pm_indices = train[train['after_pm'] == True].index
for pm_idx in pm_indices:
    ax.axvline(pm_idx, color='red', alpha=0.3, linewidth=0.8)
ax.set_xlabel('样本序号')
ax.set_ylabel('Etch Rate (nm/min)')
ax.set_title('训练集: 刻蚀速率随时间变化（红色竖线 = PM 事件）')

# PM 周期内的 aging curve
ax = axes[1]
aging_curve = train.groupby('runs_in_pm_cycle')['etch_rate'].agg(['mean', 'std'])
ax.plot(aging_curve.index, aging_curve['mean'], 'b-', linewidth=2, label='平均刻蚀速率')
ax.fill_between(aging_curve.index,
                aging_curve['mean'] - aging_curve['std'],
                aging_curve['mean'] + aging_curve['std'],
                alpha=0.2, color='blue')
ax.axvspan(0, 12, alpha=0.15, color='orange', label='Break-in 区域')
ax.set_xlabel('PM 周期内运行次数')
ax.set_ylabel('Etch Rate (nm/min)')
ax.set_title('PM 周期内的腔体老化曲线（非单调！）')
ax.legend()

plt.tight_layout()
plt.show()

print('关键观察：')
print('1. PM 后刻蚀速率先下降（break-in 阶段，腔壁太干净）')
print('2. 随后逐渐恢复并稳定')
print('3. 长期运行后因腔壁聚合物积累和 RF 老化而缓慢下降')
print('4. 这种非单调曲线是线性模型难以捕捉的')

In [ ]:
# Block 5: 传感器数据理解

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# OES H 704nm (actually F 703.7nm wavelength)
axes[0, 0].hist(train['oes_H_704_mean'], bins=50, alpha=0.7, color='#e74c3c', edgecolor='white')
axes[0, 0].set_title('OES F 704nm 强度分布')
axes[0, 0].set_xlabel('Intensity')

# OES SiF/F ratio
axes[0, 1].hist(train['oes_SiF_F_ratio'], bins=50, alpha=0.7, color='#e67e22', edgecolor='white')
axes[0, 1].set_title('SiF/F 比值分布（刻蚀产物指标）')
axes[0, 1].set_xlabel('Ratio')

# RF voltage
axes[0, 2].hist(train['rf_V_mean'], bins=50, alpha=0.7, color='#2ecc71', edgecolor='white')
axes[0, 2].set_title('RF 电压分布')
axes[0, 2].set_xlabel('Voltage')

# Pressure
axes[1, 0].hist(train['pressure_mean'], bins=50, alpha=0.7, color='#3498db', edgecolor='white')
axes[1, 0].set_title('腔体压力分布')
axes[1, 0].set_xlabel('Pressure (mTorr)')

# Power x Pressure interaction
axes[1, 1].scatter(train['power_pressure_product'], train['etch_rate'], alpha=0.3, s=5, color='#9b59b6')
axes[1, 1].set_xlabel('Power x Pressure')
axes[1, 1].set_ylabel('Etch Rate')
axes[1, 1].set_title('功率-压力交互 vs 刻蚀速率')

# CF4 flow vs etch rate
axes[1, 2].scatter(train['flow_CF4_mean'], train['etch_rate'], alpha=0.3, s=5, color='#1abc9c')
axes[1, 2].set_xlabel('CF4 Flow (sccm)')
axes[1, 2].set_ylabel('Etch Rate')
axes[1, 2].set_title('CF4 流量 vs 刻蚀速率')

plt.tight_layout()
plt.show()

print('传感器噪声来源：')
print('- OES: 散粒噪声 (8%), 1/f 等离子体漂移, 随机尖峰')
print('- RF: 高斯噪声 (3-4%), 二次谐波干扰')
print('- 压力: 控制器动态响应 (τ=0.15s), 0.3 mTorr 批次偏移')
print('- 温度: 热惯性 (τ=0.3s), 1°C 批次偏移')
print('- 气体流量: 2% MFC 噪声 + 2% 批次校准漂移')

In [ ]:
# Block 6: Train/Test 分布差异分析
from scipy.stats import ks_2samp

# 选取关键特征进行分布对比
key_features = ['oes_H_704_mean', 'rf_V_mean', 'pressure_mean', 'flow_CF4_mean',
                'power_pressure_product', 'oes_SiF_F_ratio', 'temp_mean', 'is_chamber_2']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

ks_results = []
for i, feat in enumerate(key_features):
    ax = axes[i]
    ax.hist(train[feat], bins=40, alpha=0.5, density=True, label='Train', color='steelblue')
    ax.hist(test[feat], bins=40, alpha=0.5, density=True, label='Test', color='coral')
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=8)

    stat, pval = ks_2samp(train[feat], test[feat])
    ks_results.append({'feature': feat, 'ks_stat': stat, 'p_value': pval})

plt.suptitle('训练集 vs 测试集 特征分布对比', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)
print('KS 检验结果（分布差异最大的特征）：')
print(ks_df.to_string(index=False))
print()
print('你的模型需要适应以下差异：')
print('1. 时间漂移：测试集来自更晚的时间段（lots 61-80），腔体老化状态不同')
print('2. 跨腔体：测试集包含 Chamber_2 数据，存在 tool-to-tool 差异')
print('3. 这使得简单的线性外推可能不够，需要考虑泛化能力')

In [ ]:
# Block 7: 基线模型 - PLS (Partial Least Squares)
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

# 准备数据
X_train = train[feature_cols].values
y_train = train['etch_rate'].values
X_test = test[feature_cols].values
y_test = pd.read_csv('data/test_answers.csv')['etch_rate'].values

# PLS 模型
pls = Pipeline([
    ('scaler', StandardScaler()),
    ('pls', PLSRegression(n_components=8)),
])
pls.fit(X_train, y_train)
pls_pred = pls.predict(X_test).ravel()

pls_r2 = r2_score(y_test, pls_pred)
pls_mae = mean_absolute_error(y_test, pls_pred)

print(f'PLS 结果:')
print(f'  R² = {pls_r2:.4f}')
print(f'  MAE = {pls_mae:.2f} nm/min')

# 残差分析
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 预测 vs 实际
axes[0].scatter(y_test, pls_pred, alpha=0.3, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('实际值 (nm/min)')
axes[0].set_ylabel('预测值 (nm/min)')
axes[0].set_title(f'PLS: 预测 vs 实际 (R²={pls_r2:.3f})')

# 残差分布
residuals = y_test - pls_pred
axes[1].hist(residuals, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('残差 (nm/min)')
axes[1].set_title('PLS 残差分布')

# 残差 vs 预测值
axes[2].scatter(pls_pred, residuals, alpha=0.3, s=8)
axes[2].axhline(0, color='red', linestyle='--')
axes[2].set_xlabel('预测值 (nm/min)')
axes[2].set_ylabel('残差 (nm/min)')
axes[2].set_title('PLS 残差 vs 预测值')

plt.tight_layout()
plt.show()

In [ ]:
# Block 8: 基线模型 - XGBoost
from xgboost import XGBRegressor

xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
    )),
])
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

xgb_r2 = r2_score(y_test, xgb_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)

print(f'XGBoost 结果:')
print(f'  R² = {xgb_r2:.4f}')
print(f'  MAE = {xgb_mae:.2f} nm/min')

# Feature importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

importances = xgb.named_steps['xgb'].feature_importances_
sorted_idx = np.argsort(importances)[::-1][:15]
axes[0].barh(range(15), importances[sorted_idx][::-1])
axes[0].set_yticks(range(15))
axes[0].set_yticklabels([feature_cols[i] for i in sorted_idx][::-1])
axes[0].set_xlabel('Feature Importance')
axes[0].set_title('XGBoost Top 15 特征重要性')

axes[1].scatter(y_test, xgb_pred, alpha=0.3, s=8)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('实际值')
axes[1].set_ylabel('预测值')
axes[1].set_title(f'XGBoost: 预测 vs 实际 (R²={xgb_r2:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Block 9: 基线模型 - GPR (Gaussian Process Regression)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

kernel = ConstantKernel(1.0) * Matern(nu=1.5) + WhiteKernel(noise_level=1.0)
gpr = Pipeline([
    ('scaler', StandardScaler()),
    ('gpr', GaussianProcessRegressor(
        kernel=kernel, n_restarts_optimizer=5,
        normalize_y=True, alpha=1e-4,
    )),
])

# GPR 复杂度 O(n³)，子采样到 500 个训练样本
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), 500, replace=False)
gpr.fit(X_train[idx], y_train[idx])
gpr_mean, gpr_std = gpr.predict(X_test, return_std=True)

gpr_r2 = r2_score(y_test, gpr_mean)
gpr_mae = mean_absolute_error(y_test, gpr_mean)

print(f'GPR 结果（训练子采样 500）:')
print(f'  R² = {gpr_r2:.4f}')
print(f'  MAE = {gpr_mae:.2f} nm/min')

# Uncertainty plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 排序绘图
sort_idx = np.argsort(y_test)
axes[0].scatter(range(len(y_test)), y_test[sort_idx], s=3, alpha=0.5, label='实际值')
axes[0].plot(range(len(y_test)), gpr_mean[sort_idx], 'r-', linewidth=1, label='GPR 预测')
axes[0].fill_between(range(len(y_test)),
                     gpr_mean[sort_idx] - 2*gpr_std[sort_idx],
                     gpr_mean[sort_idx] + 2*gpr_std[sort_idx],
                     alpha=0.2, color='red', label='95% 置信区间')
axes[0].set_xlabel('样本（按实际值排序）')
axes[0].set_ylabel('Etch Rate (nm/min)')
axes[0].set_title('GPR 预测与不确定性')
axes[0].legend()

axes[1].scatter(y_test, gpr_mean, alpha=0.3, s=8)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('实际值')
axes[1].set_ylabel('预测值')
axes[1].set_title(f'GPR: 预测 vs 实际 (R²={gpr_r2:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Block 10: PINN 基线
import torch
import time

# PINN: Physics-Informed MLP
# 架构: input(51) -> 64 -> 32 -> 1
# 物理约束: 第一层权重正则化（单调性）+ L2 weight decay

class PINN:
    def __init__(self, input_dim, hidden_dims=(64, 32)):
        self.input_dim = input_dim
        self.dims = [input_dim] + list(hidden_dims) + [1]
        self.scaler_X = StandardScaler()
        self.scaler_y = StandardScaler()
        self.weight_np = []
        self.bias_np = []

    def fit(self, X, y, n_epochs=8000, lr=1e-2, physics_weight=0.005):
        X_np = self.scaler_X.fit_transform(X).astype(np.float32)
        y_np = self.scaler_y.fit_transform(y.reshape(-1, 1)).ravel().astype(np.float32)

        rng_gen = torch.Generator().manual_seed(42)
        weights, biases = [], []
        for i in range(len(self.dims) - 1):
            fan_in, fan_out = self.dims[i], self.dims[i + 1]
            std = (2.0 / fan_in) ** 0.5
            w = torch.randn(fan_out, fan_in, generator=rng_gen) * std
            b = torch.zeros(fan_out)
            w.requires_grad_(True)
            b.requires_grad_(True)
            weights.append(w)
            biases.append(b)

        params = weights + biases
        optimizer = torch.optim.SGD(params, lr=lr, momentum=0.95, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=1000, T_mult=2, eta_min=1e-4
        )

        history = []
        n_samples = len(X_np)
        batch_size = 32
        t0 = time.time()

        for epoch in range(n_epochs):
            perm = np.random.permutation(n_samples)
            epoch_loss = 0.0
            n_batches = 0

            for start in range(0, n_samples, batch_size):
                idx = perm[start:start + batch_size]
                xb = torch.tensor(X_np[idx])
                yb = torch.tensor(y_np[idx])

                optimizer.zero_grad()
                h = xb
                for i, (w, b) in enumerate(zip(weights, biases)):
                    h = torch.nn.functional.linear(h, w, b)
                    if i < len(weights) - 1:
                        h = torch.relu(h)
                pred = h.squeeze(-1)

                loss = torch.nn.functional.mse_loss(pred, yb)
                # 单调性惩罚：第一层权重为正
                mono_penalty = torch.relu(-weights[0]).sum() * physics_weight
                loss = loss + mono_penalty

                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
                n_batches += 1

            scheduler.step()
            history.append(epoch_loss / n_batches)

            if epoch % 2000 == 0:
                elapsed = time.time() - t0
                print(f'  Epoch {epoch:5d}: loss={history[-1]:.6f} ({elapsed:.1f}s)')

        self.weight_np = [w.detach().numpy().copy() for w in weights]
        self.bias_np = [b.detach().numpy().copy() for b in biases]

        del weights, biases, params, optimizer, scheduler
        return history

    def predict(self, X):
        X_scaled = self.scaler_X.transform(X).astype(np.float32)
        h = X_scaled
        for i, (w, b) in enumerate(zip(self.weight_np, self.bias_np)):
            h = h @ w.T + b
            if i < len(self.weight_np) - 1:
                h = np.maximum(h, 0)
        pred = h.squeeze(-1)
        return self.scaler_y.inverse_transform(pred.reshape(-1, 1)).ravel()

# 训练 PINN
print('训练 PINN（可能需要 1-3 分钟）...')
pinn = PINN(input_dim=X_train.shape[1])
history = pinn.fit(X_train, y_train, n_epochs=8000)
pinn_pred = pinn.predict(X_test)

pinn_r2 = r2_score(y_test, pinn_pred)
pinn_mae = mean_absolute_error(y_test, pinn_pred)

print(f'\nPINN 结果:')
print(f'  R² = {pinn_r2:.4f}')
print(f'  MAE = {pinn_mae:.2f} nm/min')

# 训练 loss 曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history, linewidth=0.5, alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('PINN 训练 Loss')
axes[0].set_yscale('log')

axes[1].scatter(y_test, pinn_pred, alpha=0.3, s=8)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('实际值')
axes[1].set_ylabel('预测值')
axes[1].set_title(f'PINN: 预测 vs 实际 (R²={pinn_r2:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
# Block 11: 为什么 PINN 在这里表现不如 PLS？

print('=' * 60)
print('PINN 表现分析')
print('=' * 60)

# 1. 参数量 vs 样本量
n_params = sum(
    pinn.dims[i] * pinn.dims[i+1] + pinn.dims[i+1]
    for i in range(len(pinn.dims) - 1)
)
print(f'\n1. 参数量 vs 样本量')
print(f'   PINN 参数量: {n_params}')
print(f'   训练样本量:  {len(X_train)}')
print(f'   参数/样本比: {n_params/len(X_train):.2f}x')
print(f'   PLS 有效参数: ~{8 * (len(feature_cols) + 1)} (8 components × {len(feature_cols)+1} )')
print(f'   PLS 参数/样本比: {8 * (len(feature_cols) + 1)/len(X_train):.2f}x')
print(f'\n   结论：PINN 参数量 ({n_params}) 接近训练样本 ({len(X_train)})，')
print(f'   神经网络通常需要 5-10x 样本量才能充分训练。')

# 2. 训练 vs 测试 gap
train_pred = pinn.predict(X_train)
train_r2 = r2_score(y_train, train_pred)
print(f'\n2. 过拟合诊断')
print(f'   训练集 R²: {train_r2:.4f}')
print(f'   测试集 R²: {pinn_r2:.4f}')
print(f'   Gap:       {train_r2 - pinn_r2:.4f}')

# 3. PLS 对比
pls_train_pred = pls.predict(X_train).ravel()
pls_train_r2 = r2_score(y_train, pls_train_pred)
print(f'\n   PLS 训练 R²: {pls_train_r2:.4f}')
print(f'   PLS 测试 R²: {pls_r2:.4f}')
print(f'   PLS Gap:     {pls_train_r2 - pls_r2:.4f}')

print(f'\n3. 核心结论')
print(f'   PINN 在数据充足时（>10000 样本）可以超越 PLS，因为：')
print(f'   - 能捕捉非线性功率-压力耦合')
print(f'   - 物理约束提供外推安全性')
print(f'   - 但在 VM 场景下，通常只有 5-30 个标记样本/PM 周期')
print(f'   - 此时 PLS 的低维投影比 NN 的高维参数空间更稳定')

In [ ]:
# Block 12: 模型对比总结

results = pd.DataFrame({
    'Model': ['PLS', 'XGBoost', 'GPR', 'PINN'],
    'R²': [pls_r2, xgb_r2, gpr_r2, pinn_r2],
    'MAE': [pls_mae, xgb_mae, gpr_mae, pinn_mae],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

bars = axes[0].bar(results['Model'], results['R²'], color=colors)
axes[0].set_ylabel('R² Score')
axes[0].set_title('模型 R² 对比')
axes[0].set_ylim(0, 1)
for bar, val in zip(bars, results['R²']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=11)

bars = axes[1].bar(results['Model'], results['MAE'], color=colors)
axes[1].set_ylabel('MAE (nm/min)')
axes[1].set_title('模型 MAE 对比')
for bar, val in zip(bars, results['MAE']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()

print('模型对比总结:')
print(results.to_string(index=False))
print()
print('各模型优缺点:')
print('┌──────────┬──────────────────────────────┬──────────────────────────────┐')
print('│ 模型     │ 优点                         │ 缺点                         │')
print('├──────────┼──────────────────────────────┼──────────────────────────────┤')
print('│ PLS      │ 快速、稳定、可解释           │ 线性假设，非线性能力有限     │')
print('│ XGBoost  │ 自动特征选择，非线性         │ 外推能力差，需要调参         │')
print('│ GPR      │ 不确定性量化，灵活核函数     │ O(n³) 复杂度，训练集受限     │')
print('│ PINN     │ 物理约束，外推安全           │ 数据量不足时欠拟合           │')
print('└──────────┴──────────────────────────────┴──────────────────────────────┘')

In [ ]:
# Block 13: 提交代碼模板

# === 提交格式 ===
# 你的提交文件应包含 run_id 和预测的 etch_rate 两列

def make_submission(predictions, filename='my_submission.csv'):
    """生成提交文件。
    
    Args:
        predictions: 长度为 len(test) 的数组，预测的 etch_rate
        filename: 输出文件名
    """
    submission = pd.DataFrame({
        'run_id': test['run_id'].values,
        'etch_rate': predictions,
    })
    submission.to_csv(filename, index=False)
    print(f'提交文件已保存: {filename} ({len(submission)} 行)')
    return submission

# === 评估函数 ===
def evaluate(predictions):
    """评估预测结果（仅供自测，实际评分由系统完成）."""
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    print(f'R² = {r2:.4f}, MAE = {mae:.2f} nm/min')
    return r2, mae

# === 示例：用 PLS 生成提交 ===
print('示例提交（PLS 基线）:')
evaluate(pls_pred)
sub = make_submission(pls_pred, 'submission_pls_baseline.csv')
sub.head()

## Block 14: 改进方向提示

### 特征工程
- 非线性变换：对高相关性特征取 log、sqrt、平方项
- 多项式特征：`sklearn.preprocessing.PolynomialFeatures`（注意维度爆炸）
- 分箱（binning）：将 `runs_in_pm_cycle` 分为 break-in / stable / aging 三段
- 特征选择：移除低相关性特征，减少噪声维度

### 半监督学习
- 训练集中只有 12% 有 metrology 标签，但所有数据都有特征
- 可以用自编码器（Autoencoder）学习特征表示，再在有标签数据上微调
- 或用伪标签（pseudo-labeling）：先用强模型预测无标签数据，再加入训练

### 迁移学习 / 域适应
- 训练集只有 Chamber_1，测试集有 Chamber_2
- 可以用域适应（Domain Adaptation）方法对齐两个腔体的特征分布
- 或者在特征中加入腔体交互项

### 集成方法
- PLS + XGBoost 融合：线性 + 非线性互补
- Stacking：用多个基模型的预测作为元特征
- 加权平均：根据验证集表现分配权重

### PINN 改进方向
- **缩小网络**：尝试 (32, 16) 或 (16, 8)，减少参数量
- **增加数据**：利用数据增强（添加噪声）或半监督方法
- **迁移学习**：先在大数据集（私有/仿真）上预训练，再在小数据上微调，不適用本比賽但或許適用解決此類問題
- **更强的物理约束**：加入能量守恒、质量守恒等硬约束

### 高级方向
- **概念漂移检测**：检测 PM 事件后的分布变化，动态调整模型
- **在线学习**：随着新数据到来，增量更新模型
- **贝叶斯神经网络**：比 dropout MC 更好的不确定性估计

---

**比赛宗旨：** 不只是追求最高 R²，更重要的是理解数据、分析问题、提出合理的改进方案。
方法论评审占 20%——好的分析和思路比盲目调参更有价值！